# RAG Pipeline — Vanilla RAG

Baseline retrieval-augmented generation over a theoretical physics PhD thesis on 3d mirror symmetry. Uses page-level chunking, BGE-large embeddings, and Mistral-7B as the reader LLM.

## 1. Installation

In [ ]:
!pip install -Uq huggingface_hub
!pip install -Uq llama-index
!pip install -Uq pymupdf llama-index-readers-file
!pip install -Uq llama-index-embeddings-huggingface
!pip install -Uq llama-index-llms-huggingface
!pip install -Uq bitsandbytes
!pip install -Uq nest_asyncio

## 2. Imports and Initialisation

In [ ]:
import os
import re
import json
import random
import numpy as np
import torch
import nest_asyncio

from google.colab import drive, userdata
from huggingface_hub import login

nest_asyncio.apply()
drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 3. Data Extraction

Load the thesis PDF using PyMuPDF. Returns one `Document` per page (~344 pages).

In [ ]:
from llama_index.readers.file import PyMuPDFReader
from llama_index.core import Document

loader = PyMuPDFReader()
docs = loader.load(file_path="/content/drive/MyDrive/SISSA_Thesis_fin_promise.pdf")
print(f"Pages loaded: {len(docs)}")

Spot-check a page with equations:

In [ ]:
print(docs[51].text[:2000])

## 4. Indexing

Embed all pages using `BAAI/bge-large-en-v1.5` and build a FAISS vector index. Runs on GPU. Takes ~20 minutes on a T4.

In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",
    device="cuda"
)

index = VectorStoreIndex.from_documents(docs, show_progress=True)

In [ ]:
# Persist index to Drive so it can be reloaded without re-embedding
index.storage_context.persist(persist_dir="/content/drive/MyDrive/rag_index")
print("Index saved.")

### Reload index (use this cell after a runtime restart instead of re-embedding)

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",
    device="cuda"
)

storage_context = StorageContext.from_defaults(
    persist_dir="/content/drive/MyDrive/rag_index"
)
index = load_index_from_storage(storage_context)
print("Index loaded.")

## 5. Reader LLM

Mistral-7B-Instruct-v0.3, 4-bit quantised via bitsandbytes. Runs close to T4 memory limits (~13GB). Load after the index to avoid OOM.

In [ ]:
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import PromptTemplate
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

Settings.llm = HuggingFaceLLM(
    model_name="mistralai/Mistral-7B-Instruct-v0.3",
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.3",
    device_map="cuda",
    model_kwargs={"quantization_config": quantization_config}
)

qa_prompt = PromptTemplate(
    "You are a precise physics assistant. Answer ONLY using the context below. "
    "Do not use any prior knowledge. If the answer is not in the context, say 'Not found in context'.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)

query_engine = index.as_query_engine(
    similarity_top_k=5,
    text_qa_template=qa_prompt
)
print("Query engine ready.")

## 6. Retrieval Evaluation

Gold set of 10 domain-expert questions. Evaluate whether the correct thesis page appears in the top-5 retrieved chunks.

In [ ]:
questions = [
    "In the fundamental N=2 mirror pair, what are the three operators mapped across the duality, and what is the superpotential on the mirror side?",
    "When a single fermion of positive mass is integrated out in a U(1) gauge theory, what happens to the CS level and what gravitational term is generated?",
    "In the planar mirror of U(N) SQCD, what is the UV topological symmetry of the planar quiver and what does it enhance to in the IR?",
    "In Figure 1.1, why does applying the S-element of SL(2;Z) to the Hanany-Witten configuration not immediately give a supersymmetric mirror, and what procedure restores supersymmetry?",
    "In Appendix B, after performing all column dualizations, how many singlets are generated in total, and what representation of the flavor symmetry do they reproduce?",
    "In the USp phase diagram, what are the qualitatively distinct zones in the (F, |k|) plane, and what distinguishes the planar duals in each zone?",
    "Which three theories are identified as marginally chiral, and what is the special feature of their planar mirror duals?",
    "What happens when a single scalar field is integrated out of SU(2) QCD3 with F scalar fields, and does the resulting duality match the general proposal?",
    "What two inequivalent limits can be defined for reducing three dimensional mirrors to two dimensional mirrors?",
    "The thesis contains a footnote stating that the notation S in Class S theories stands for something the author finds disappointing. What does it stand for and why is it mentioned?",
]

retriever = index.as_retriever(similarity_top_k=5)

for i, q in enumerate(questions):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {q}")
    print('='*60)
    nodes = retriever.retrieve(q)
    for j, node in enumerate(nodes):
        print(f"  Chunk {j+1} | Page {node.metadata['source']} | Score {node.score:.3f}")
        print(f"  {node.text[:200]}")
        print()

### Baseline retrieval results

| Question | Correct page | In top 5? |
|----------|-------------|----------|
| Q1 (operator map) | 315 | ✗ |
| Q2 (CS level shift) | 52/53 | ✓ |
| Q3 (topological symmetry) | 130 | ✓ |
| Q4 (S-rule, HW moves) | 77/78 | ✓ |
| Q5 (singlet count) | Appendix B | ✗ |
| Q6 (USp zones) | 216 | ✓ |
| Q7 (marginally chiral) | 164 | ✓ |
| Q8 (scalar integrated out) | 285/287 | ✓ |
| Q9 (2d limits) | 305 | ✓ |
| Q10 (Class S footnote) | 313 | ✗ |
| **Score** | | **7/10** |